<a href="https://colab.research.google.com/github/Dr-Isam-ALJAWARNEH/fds-project-geollms/blob/LLM-integration/LLM_integration.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip install groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.5/127.5 kB 3.4 MB/s eta 0:00:00


In [ ]:
import os
import pandas as pd

# Step 1: Clone the GitHub repository if not already present
repo_url = "https://github.com/Dr-Isam-ALJAWARNEH/fds-project-geollms.git"
clone_dir = "fds-project-geollms"

if not os.path.exists(clone_dir):
    os.system(f"git clone {repo_url}")

# Step 2: Path to NDVI folder
ndvi_folder = os.path.join(clone_dir, "Datasets", "NDVI")

# Step 3: Read and analyze each CSV
all_dfs = []
print("Reading files from NDVI folder...\n")
for file in os.listdir(ndvi_folder):
    if file.endswith(".csv"):
        file_path = os.path.join(ndvi_folder, file)
        df = pd.read_csv(file_path)
        all_dfs.append(df)
        print(f"File: {file}")
        print(f"    Rows: {len(df)}, Columns: {len(df.columns)}\n")

# Step 4: Combine all into one DataFrame
combined_ndvi_df = pd.concat(all_dfs, ignore_index=True)

# Step 5: Summary of combined data
print("Combined NDVI Dataset:")
print(f"    Total Rows: {len(combined_ndvi_df)}")
print(f"    Total Columns: {len(combined_ndvi_df.columns)}")
print("\n First 5 rows of the combined dataset:")
print(combined_ndvi_df.head())

# Check for missing values
missing_ndvi = combined_ndvi_df['grid_code'].isnull().sum()
missing_datetime = combined_ndvi_df['Date'].isnull().sum()

# Display results
print(f"Missing values in 'grid_code': {missing_ndvi}")
print(f"Missing values in 'Date': {missing_datetime}")
print(f"Total number of rows after combining: {combined_ndvi_df.shape[0]}")

# Print all column names
print("Columns in the combined dataset:")
print(combined_ndvi_df.columns.tolist())


Reading files from NDVI folder...

File: NDVI20221108.csv
    Rows: 1048575, Columns: 6

File: NDVI20220719.csv
    Rows: 1048575, Columns: 6

File: NDVI20220116.csv
    Rows: 1048575, Columns: 6

File: NDVI20230212.csv
    Rows: 1048575, Columns: 6

File: NDVI20211106.csv
    Rows: 1048575, Columns: 6

File: NDVI20221015.csv
    Rows: 1048575, Columns: 6

File: NDVI20220609.csv
    Rows: 1048575, Columns: 6

File: NDVI20211122.csv
    Rows: 1048575, Columns: 6

File: NDVI20230204.csv
    Rows: 1048575, Columns: 6

File: NDVI20220226.csv
    Rows: 1048575, Columns: 6

File: NDVI20220703.csv
    Rows: 1048575, Columns: 6

File: NDVI20221023.csv
    Rows: 1048575, Columns: 6

File: NDVI20210825.csv
    Rows: 1048575, Columns: 6

File: NDVI20220617.csv
    Rows: 1048575, Columns: 6

File: NDVI20210919.csv
    Rows: 1048575, Columns: 6

File: NDVI20210701.csv
    Rows: 1048575, Columns: 6

File: NDVI20221218.csv
    Rows: 1048575, Columns: 6

File: NDVI20220321.csv
    Rows: 1048575, Colum

In [ ]:
import pandas as pd
import requests
from io import StringIO

# URL for GitHub files
base_url = "https://raw.githubusercontent.com/Dr-Isam-ALJAWARNEH/fds-project-geollms/main/Datasets/AQ_data/"

# List of files to read
filenames = [f"chicago_eclipse_data_part_{i}.csv" for i in range(1, 20)]

# List to store DataFrames
dfs = []

# Download and read each CSV into a DataFrame
for filename in filenames:
    url = base_url + filename
    response = requests.get(url)
    if response.status_code == 200:
        df = pd.read_csv(StringIO(response.text))
        dfs.append(df)
    else:
        print(f"Failed to load: {filename}")

# Combine all CSV files into one DataFrame
combined_df = pd.concat(dfs, ignore_index=True)

# Check for missing values
missing_pm25 = combined_df['PM25'].isnull().sum()
missing_datetime = combined_df['ReadingDateTimeUTC'].isnull().sum()

# Display results
print(f"Missing values in 'PM25': {missing_pm25}")
print(f"Missing values in 'ReadingDateTimeUTC': {missing_datetime}")
print(f"Total number of rows after combining: {combined_df.shape[0]}")

# Print all column names
print("Columns in the combined dataset:")
print(combined_df.columns.tolist())


Missing values in 'PM25': 0
Missing values in 'ReadingDateTimeUTC': 0
Total number of rows after combining: 2461089
Columns in the combined dataset:
['City', 'DeviceId', 'LocationName', 'Latitude', 'Longitude', 'ReadingDateTimeUTC', 'PM25', 'CalibratedPM25', 'CalibratedO3', 'CalibratedNO2', 'CO', 'Temperature', 'Humidity', 'BatteryLevel', 'PercentBattery', 'CellSignal']


In [ ]:
import json
from groq import Groq

# Initialize Groq client
client = Groq(api_key="YOUR_API")

# Define the LLM prompt for extraction
system_prompt = '''Extract origin, destination, and route preference from the user's travel request. Handle typos and missing spaces if present.
Valid preferences: "shortest", "healthiest", or "greenest" or combination of multiple of these eg. "shortest and healthiest", "shortest and healthiest and greenest".
Return ONLY the result in valid JSON format.
{
  "origin": "origin name",
  "destination": "destination name",
  "preference": "shortest | healthiest | greenest"
}'''

# Define test prompts covering different combinations and typos
test_prompts = [
    "I want to go from Melrose Park to Hyde Park using the healthiest route",
    "Give me the greenest route from Lincoln Park to Garfield Park",
    "I need the shortest route between Loop and Hyde Park",
    "Please find the healthiest and greenest route from Melrose Park to Loop",
    "I'm going from Lincoln Park to Loop, I want the healthiest and shortest path",
    "Travel from Hyde Park to Garfield Park using the shortest and greenest route",
    "Melrose Park to Lincoln Park healthiest shortest greenest route please",
    "Go from HydePark to MelrosePark healthiest option",
    "How can I bike from Navy Pier to Hyde Park with the cleanest air?",
    "show me a route from Millennium Park to Logan Square avoiding polluted areas",
    "I want the most health-friendly route from Lincoln Park to Chinatown",
    "bike path from Bucktown to Loop with least pm25",
    "go from West Loop to Garfield Park with best air today",
    "What's the most scenic bike path from Wicker Park to the Museum Campus?",
    "Can I go from Uptown to South Shore through green areas?",
    "route from Roseland to Bronzeville with lots of trees",
    "i want to ride from Andersonville to downtown using the greenest way",
    "Bike me from Edgewater to Pilsen through leafy roads",
    "What's the shortest bike route from Loop to Lincoln Square?",
    "How quick is the ride from West Loop to Old Town?",
    "shortest biking path from South Side to North Side chicago",
    "fastest way from O'Hare to Logan Square by bike?",
    "Need shortest bike road from Humboldt park to Near North",
    "i wanna go frm lakeview to the field musuem green plz",
    "go from chinatown to uic campus w/ clean air n trees",
    "show me bike path from oak park to west loop (safe?)",
    "Best cycle way from McKinley park to the art institute healthy one plz",
    "hey i wanna bike frm rogers park to the loop — shortest & clean route",
    "wanna go biking from greektown to lincoln park with clean air and trees",
    "ride frm south loop to oakwood — prefer healthiest path pls",
    "i wanna go from logan sqr to the musuem campus – cleanest air pls",  # typo: "sqr" and "musuem"
    "bike route frm willies tower to grant pak, make it the shortest",   # typos: "willies" (Willis), "pak" (Park)
    "show me a path from south loop to boystown with trees n good air",  # typo-ish name: "boystown" (local nickname for Lakeview East)
    "frm brigdeport to linclon park — i want trees and clean air",       # typos: "brigdeport", "linclon"
    "give me a route from humblt park to sheed aquarim that’s green, shortest, and healthiest"  # typo: "humblt", "sheed aquarim", complex criteria
]

# Function to query the LLM
def extract_trip_info(prompt):
    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": prompt}
        ],
        temperature=0
    )
    return json.loads(response.choices[0].message.content)

# Run all test prompts and print results
for i, prompt in enumerate(test_prompts, start=1):
    print(f"\n--- Test Case {i} ---")
    print(f"Prompt: {prompt}")
    try:
        result = extract_trip_info(prompt)
        print("Extracted JSON:", json.dumps(result, indent=2))
    except Exception as e:
        print("Error parsing JSON:", str(e))



--- Test Case 1 ---
Prompt: I want to go from Melrose Park to Hyde Park using the healthiest route
Extracted JSON: {
  "origin": "Melrose Park",
  "destination": "Hyde Park",
  "preference": "healthiest"
}

--- Test Case 2 ---
Prompt: Give me the greenest route from Lincoln Park to Garfield Park
Extracted JSON: {
  "origin": "Lincoln Park",
  "destination": "Garfield Park",
  "preference": "greenest"
}

--- Test Case 3 ---
Prompt: I need the shortest route between Loop and Hyde Park
Extracted JSON: {
  "origin": "Loop",
  "destination": "Hyde Park",
  "preference": "shortest"
}

--- Test Case 4 ---
Prompt: Please find the healthiest and greenest route from Melrose Park to Loop
Extracted JSON: {
  "origin": "Melrose Park",
  "destination": "Loop",
  "preference": "healthiest and greenest"
}

--- Test Case 5 ---
Prompt: I'm going from Lincoln Park to Loop, I want the healthiest and shortest path
Extracted JSON: {
  "origin": "Lincoln Park",
  "destination": "Loop",
  "preference": "heal